In [1]:
import polars as pl
import os
import glob
from tqdm import tqdm

## Functions

In [2]:
def reorder_cols(df: pl.DataFrame) -> pl.DataFrame:
	"""
	Reorders the columns of a DataFrame so that 'sample' is the first column if it exists.
	"""
	cols = df.columns

	if "sample" in cols:
		cols.remove("sample")
		cols.insert(0, "sample")

	return df[cols]

In [3]:
def snv_filter(df: pl.DataFrame):
    return df.filter(
        (pl.col("ref").str.len_chars() == 1) &
        (pl.col("alt").str.len_chars() == 1)
	)

In [4]:
def summarize_res(df: pl.DataFrame) -> pl.DataFrame:
	
	bool_cols = df['filter_1_mutation_intra_hairpin_loop':'filter_8_low_quality'].columns
	str_cols = df['msec_filter_123':'msec_filter_all'].columns
	n_variants = df.height

	# Initialize a dictionary
	results = {
		"filter": [],
		"percentage": [],
		"n_failed": []
	}

	# Populate the dictionary inside the loops
	for col in bool_cols:
		n_failed = df[col].sum() # Sum counts True values
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	for col in str_cols:
		n_failed = df.filter(~pl.col(col).is_null()).height # Count non-nulls
		
		results["filter"].append(col)
		results["percentage"].append((n_failed / n_variants) * 100)
		results["n_failed"].append(n_failed)

	# Create DataFrame from the dictionary
	return pl.DataFrame(results).with_columns(pl.lit(n_variants).alias("total_variants"))

### MicroSEC filter Description

The MicroSEC pipeline contains 8 filtering processes.  

- Filter 1  : Shorter-supporting lengths distribute too short to occur (1-1 and 1-2).  
	- Filter 1-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 1-2: The shorter-supporting lengths distributed over less than 75% of the read length.  
- Filter 2  : Hairpin-structure induced error detection (2-1 and 2-2).  
	- Filter 2-1: Palindromic sequences exist within 200 bases.  
	- Filter 2-2: >=50% mutation-supporting reads contains a reverse complementary sequence of the opposite strand consisting >= 15 bases.  
- Filter 3  : 3'-/5'-supporting lengths are too densely distributed to occur (3-1 and 3-2).  
	- Filter 3-1: P-values are less than the threshold_p(default: 10^(-6)).  
	- Filter 3-2: The distributions of 3'-/5'-supporting lengths are within 75% of the read length.  
- Filter 4  : >=15% mutations were called by chimeric reads comprising two distant regions.  
- Filter 5  : >=50% mutations were called by soft-clipped reads.  
- Filter 6  : Mutations locating at simple repeat sequences.  
- Filter 7  : Indel mutations locating at a >=15 homopolymer.  
- Filter 8  : >=10% of bases are low quality (Quality score <18) in the mutation supporting reads.  

Filter 1, 2, 3, and 4 detect possible FFPE artifacts.  
Filter 5 may also be FFPE artifacts or mapping errors.  
Filter 6, 7, and 8 detect frequent errors caused by the next generation sequencing platform.  
Supporting lengths are adjusted considering small repeat sequences around the mutations.  
  
Results are saved in a tsv file.  

github url: https://github.com/MANO-B/MicroSEC

## Main
### VCF

In [5]:
msec_paths = sorted(glob.glob("../vcf-micr-svf/*/*.microsec.tsv"))

all_res = []

for path in tqdm(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t", infer_schema_length = 1000).rename(lambda x : x.lower())
	
	all_res.append(df)
	
vcf_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1641/1641 [02:14<00:00, 12.21it/s]


In [6]:
vcf_df

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""12-del""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1001159-01""","""1-snv""","""chr1""",16262742,"""G""","""A""","""N""","""CTCTGTTGGCCTGCCTTCCCAGACCAAGAC…",49,772,2,0,48,48,24,524,399,0.012821,0.015803,0.017098,0.0,0.002591,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr1""",45797760,"""T""","""C""","""N""","""AGAGCTGTTCCTGCTCCACCCGAGAGGCAC…",49,928,4,0,48,48,24,490,415,0.015306,0.019935,0.014763,0.0,0.00431,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr10""",43601830,"""G""","""A""","""N""","""TCTGCATCCTGCAGGACACCATGGTGGCCA…",49,451,1,0,48,48,24,355,380,0.011675,0.013969,0.013969,0.0,0.002217,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""1-snv""","""chr11""",118307334,"""C""","""T""","""N""","""CGCCCCGCGGCAACGCGTCCTGGCCCTGCT…",49,34,0,0,46,45,23,174,88,0.006603,0.008824,0.008824,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""1-snv""","""chr17""",37856504,"""G""","""A""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""1-snv""","""chr22""",29108003,"""C""","""T""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""1-snv""","""chr4""",1920021,"""A""","""C""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null


In [7]:
## Samples that could be processed
vcf_df["sample"].unique()

sample
str
"""ORD-1616957-01"""
"""ORD-1977243-01"""
"""ORD-1407403-01"""
"""ORD-2099386-01"""
"""ORD-1593214-01"""
…
"""ORD-2016906-01"""
"""ORD-2093178-01"""
"""ORD-1721357-01"""


In [8]:
# All Filters
vcf_arti_all_filter = vcf_df.filter(~pl.col("msec_filter_all").is_null()).pipe(snv_filter)
display(vcf_arti_all_filter)
# Number of Samples with >= 1 artifacts
vcf_arti_all_filter["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1048486-01""","""1-snv""","""chr4""",55140704,"""G""","""A""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1049250-01""","""1-snv""","""chr11""",69518363,"""G""","""C""","""Y""","""TGGGGTGGGGCGCGCGCAGCCGGGTGGGGT…",49,367,3,0,48,48,24,452,395,0.019018,0.016076,0.023706,0.0,0.008174,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1050682-01""","""1-snv""","""chr19""",11098593,"""G""","""T""","""N""","""TGGAGATCCTGCAGGAGCGCTAGTACAGGT…",144,27,16,0,129,90,59,293,295,0.006944,0.018519,0.007407,0.0,0.592593,0.000011,0.000003,0.000002,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1050682-01""","""1-snv""","""chr5""",79950719,"""C""","""T""","""Y""","""TGCAGCGGCTGCAGCGGCCGTAGCGGCCGC…",144,2994,376,0,143,142,71,387,325,0.018653,0.014729,0.025518,0.0,0.125585,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1075235-01""","""1-snv""","""chr11""",118369282,"""A""","""G""","""N""","""CTTTAAAAAAAAAAAAAAAGGCTTTTTTAG…",49,173,35,0,48,47,24,395,190,0.074437,0.010405,0.202312,0.0,0.202312,1.0,1.0,1.0,false,false,false,false,false,false,true,true,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2130093-01""","""1-snv""","""chrX""",63411335,"""C""","""A""","""Y""","""CTTCCCAAGTGTGGGCCTCCATGGCATAGG…",101,404,29,0,100,100,49,184,228,0.008774,0.011386,0.010891,0.0,0.071782,0.016633,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-2130099-01""","""1-snv""","""chrX""",63411410,"""C""","""T""","""Y""","""CCTCCCTGGCATGAGCTTCTTGGGCACGTG…",101,128,1,0,100,100,50,393,405,0.019261,0.015625,0.01875,0.0,0.0078125,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""1-snv""","""chr13""",48954189,"""G""","""T""","""N""","""TTTTAAATTATCTGTTTCAGTAAGAAGAAC…",144,1511,477,0,145,144,71,477,626,0.122082,0.15182,0.150827,0.0,0.315685,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null


919

In [9]:
# Filter 1234
vcf_arti_filter_1234 = vcf_df.filter(~pl.col("msec_filter_1234").is_null()).pipe(snv_filter)
display(vcf_arti_filter_1234)
# Number of Samples with >= 1 artifacts
vcf_arti_filter_1234["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1048486-01""","""1-snv""","""chr4""",55140704,"""G""","""A""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1084423-01""","""1-snv""","""chr22""",29689746,"""G""","""A""","""N""","""GTATTTTTAGTAGAGATGGGATTTGGTGAA…",49,22,0,0,45,48,22,512,84,0.008349,0.0,0.004545,0.181818,0.0,0.528493,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1087392-01""","""1-snv""","""chr2""",47643016,"""G""","""T""","""N""","""CAGGAGTTGAGACCAGCCTGTGCAACATAG…",49,17,0,0,47,46,21,252,375,0.009604,0.005882,0.005882,0.294118,0.0,0.075877,0.441823,0.441823,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1111864-01""","""1-snv""","""chr11""",118354217,"""C""","""T""","""Y""","""GAGATCGCGCCACTGCACTCTAGCTTGGGT…",49,81,1,0,47,48,24,47,265,0.038549,0.071605,0.030864,0.209877,0.012346,1.0,0.568666,0.568666,false,false,false,true,false,true,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1114063-01""","""1-snv""","""chr2""",212288966,"""C""","""T""","""N""","""CTAATAAATCAGGGATTTCTTGCGTTGGAA…",144,38,14,0,90,133,71,242,133,0.027047,0.034211,0.042105,0.0,0.368421,0.066392,1.7303e-8,3.0432e-8,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2115040-01""","""1-snv""","""chr16""",3778242,"""C""","""T""","""N""","""GCCCCATCTGGCCAAGCTGTTCCATCTGAG…",101,19,0,0,90,79,21,90,79,0.007817,0.005263,0.010526,0.0,0.0,1.4551e-12,0.000022,0.000023,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2115040-01""","""1-snv""","""chrX""",63411333,"""C""","""T""","""Y""","""AGCTTCCCAAGTGTGGGCCTTCCTGGCATA…",101,22,0,0,17,83,17,17,83,0.020702,0.009091,0.018182,0.0,0.0,2.4810e-33,5.0376e-38,5.0376e-38,true,false,true,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2117808-01""","""1-snv""","""chr3""",178952085,"""A""","""G""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null


410

In [10]:
summarize_res(vcf_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.914739,424,46352
"""filter_2_hairpin_structure""",0.030204,14,46352
"""filter_3_microhomology_induced…",1.382896,641,46352
"""filter_4_highly_homologous_reg…",1.279341,593,46352
"""filter_5_soft_clipped_reads""",0.509147,236,46352
…,…,…,…
"""filter_7_mutation_at_homopolym…",2.524163,1170,46352
"""filter_8_low_quality""",3.21669,1491,46352
"""msec_filter_123""",1.876942,870,46352


In [11]:
summarize_res(vcf_arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",20.478326,137,669
"""filter_2_hairpin_structure""",0.149477,1,669
"""filter_3_microhomology_induced…",30.044843,201,669
"""filter_4_highly_homologous_reg…",62.481315,418,669
"""filter_5_soft_clipped_reads""",5.082212,34,669
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.44843,3,669
"""filter_8_low_quality""",7.473842,50,669
"""msec_filter_123""",37.967115,254,669


In [12]:
summarize_res(vcf_arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",8.002336,137,1712
"""filter_2_hairpin_structure""",0.058411,1,1712
"""filter_3_microhomology_induced…",11.740654,201,1712
"""filter_4_highly_homologous_reg…",24.415888,418,1712
"""filter_5_soft_clipped_reads""",8.703271,149,1712
…,…,…,…
"""filter_7_mutation_at_homopolym…",3.621495,62,1712
"""filter_8_low_quality""",35.046729,600,1712
"""msec_filter_123""",14.836449,254,1712


### XML

In [13]:
msec_xml_paths = sorted(glob.glob("../xml-micr-svf/*/*.microsec.tsv"))
len(msec_xml_paths)

1606

In [14]:
all_res = []

for path in tqdm(msec_xml_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t", infer_schema_length = 1000).rename(lambda x: x.lower())
	
	all_res.append(df)
	
xml_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

100%|██████████| 1606/1606 [01:12<00:00, 22.24it/s]


In [15]:
xml_df

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1001159-01""","""chr1""",11190697,"""CGTGGTGGCGGCA""","""C""","""MTOR""",true,1053,"""5490_5501delTGCCGCCACCAC""","""T1834_T1837del""",0.3922,"""nonframeshift""","""NM_004958""","""-""",false,"""12-del""","""Y""","""GTGGTGGTGGCAGTGGCGGCCGTGGTGGCG…",49,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1001159-01""","""chr1""",16262742,"""G""","""A""","""SPEN""",true,902,"""10007G>A""","""R3336Q""",0.5488,"""missense""","""NM_015001""","""+""",false,"""1-snv""","""N""","""CTCTGTTGGCCTGCCTTCCCAGACCAAGAC…",49,772,2,0,48,48,24,524,399,0.012821,0.015803,0.017098,0.0,0.002591,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr1""",45797760,"""T""","""C""","""MUTYH""",false,1044,"""892-2A>G""","""splice site 892-2A>G""",0.5354,"""splice""","""NM_001048171""","""-""",false,"""1-snv""","""N""","""AGAGCTGTTCCTGCTCCACCCGAGAGGCAC…",49,928,4,0,48,48,24,490,415,0.015306,0.019935,0.014763,0.0,0.00431,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr10""",43601830,"""G""","""A""","""RET""",true,631,"""874G>A""","""V292M""",0.4517,"""missense""","""NM_020975""","""+""",false,"""1-snv""","""N""","""TCTGCATCCTGCAGGACACCATGGTGGCCA…",49,451,1,0,48,48,24,355,380,0.011675,0.013969,0.013969,0.0,0.002217,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1001159-01""","""chr11""",118307334,"""C""","""T""","""MLL""",true,31,"""107C>T""","""P36L""",0.9032,"""missense""","""NM_005933""","""+""",false,"""1-snv""","""N""","""CGCCCCGCGGCAACGCGTCCTGGCCCTGCT…",49,34,0,0,46,45,23,174,88,0.006603,0.008824,0.008824,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""chr17""",37856504,"""G""","""A""","""ERBB2""",true,3682,"""13G>A""","""A5T""",0.0019,"""missense""","""NM_004448""","""+""",false,"""1-snv""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""chr22""",29108003,"""C""","""T""","""CHEK2""",true,3729,"""686G>A""","""G229D""",0.0013,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""chr4""",1920021,"""A""","""C""","""WHSC1""",true,1979,"""1081A>C""","""K361Q""",0.5073,"""missense""","""NM_133335""","""+""",false,"""1-snv""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,fal

In [16]:
## Samples that could be processed
xml_df["sample"].unique()

sample
str
"""ORD-1974403-01"""
"""ORD-1888385-01"""
"""ORD-1246850-01"""
"""ORD-1884020-01"""
"""ORD-1939379-01"""
…
"""ORD-1094680-01"""
"""ORD-1462572-01"""
"""ORD-1504082-01"""


In [17]:
# All artifacts
xml_arti_all_filter = xml_df.filter(~pl.col("msec_filter_all").is_null()).pipe(snv_filter)
display(xml_arti_all_filter)
# Number of Samples with >= 1 artifacts
xml_arti_all_filter["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1048486-01""","""chr4""",55140704,"""G""","""A""","""PDGFRA""",true,3073,"""1565G>A""","""R522H""",0.0026,"""missense""","""NM_006206""","""+""",false,"""1-snv""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1050682-01""","""chr19""",11098593,"""G""","""T""","""SMARCA4""",false,960,"""1111G>T""","""E371*""",0.0063,"""nonsense""","""NM_003072""","""+""",false,"""1-snv""","""N""","""TGGAGATCCTGCAGGAGCGCTAGTACAGGT…",144,27,16,0,129,90,59,293,295,0.006944,0.018519,0.007407,0.0,0.592593,0.000011,0.000003,0.000002,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1050682-01""","""chr5""",79950719,"""C""","""T""","""MSH3""",false,488,"""173C>T""","""A58V""",0.5184,"""missense""","""NM_002439""","""+""",false,"""1-snv""","""Y""","""TGCAGCGGCTGCAGCGGCCGTAGCGGCCGC…",144,2994,376,0,143,142,71,387,325,0.018653,0.014729,0.025518,0.0,0.125585,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1075235-01""","""chr9""",98278972,"""T""","""C""","""PTCH1""",false,487,"""131A>G""","""E44G""",0.5975,"""missense""","""NM_001083603""","""-""",false,"""1-snv""","""Y""","""CTCCGTTTTCTTCTTCTTCTCCTCCTCCTC…",49,668,3,0,48,48,24,314,425,0.015917,0.018413,0.014671,0.0,0.004491,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1094680-01""","""chr19""",15376352,"""G""","""A""","""BRD4""",true,1717,"""662C>T""","""T221M""",0.4619,"""missense""","""NM_014299""","""-""",false,"""1-snv""","""N""","""CGGCAGGGAAGGGGTGAGGCATGGCCTGCA…",144,8804,1936,0,147,143,72,574,533,0.051305,0.014493,0.133167,0.0,0.2199,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2130093-01""","""chrX""",63411335,"""C""","""A""","""FAM123B""",true,174,"""1832G>T""","""R611M""",0.5115,"""missense""","""NM_152424""","""-""",false,"""1-snv""","""Y""","""CTTCCCAAGTGTGGGCCTCCATGGCATAGG…",101,404,29,0,100,100,49,184,228,0.008774,0.011386,0.010891,0.0,0.071782,0.016633,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-2130099-01""","""chrX""",63411410,"""C""","""T""","""FAM123B""",true,853,"""1757G>A""","""R586Q""",0.0563,"""missense""","""NM_152424""","""-""",false,"""1-snv""","""Y""","""CCTCCCTGGCATGAGCTTCTTGGGCACGTG…",101,128,1,0,100,100,50,393,405,0.019261,0.015625,0.01875,0.0,0.0078125,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""chr13""",48954189,"""G""","""T""","""RB1""",false,2060,"""1390G>T""","""E464*""",0.0733,"""nonsense""","""NM_000321""","""+""",false,"""1-s

574

In [18]:
# Filter 1234
xml_arti_filter_1234 = xml_df.filter(~pl.col("msec_filter_1234").is_null()).pipe(snv_filter)
display(xml_arti_filter_1234)
# Number of Samples with >= 1 artifacts
xml_arti_filter_1234["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1048486-01""","""chr4""",55140704,"""G""","""A""","""PDGFRA""",true,3073,"""1565G>A""","""R522H""",0.0026,"""missense""","""NM_006206""","""+""",false,"""1-snv""","""N""","""CTCTTGTCACGTAGCCCTGCATTCTGAACT…",144,54,34,0,68,115,68,170,345,0.00643,0.005556,0.007407,0.0,0.62963,0.000311,1.0089e-9,1.5114e-18,false,false,true,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1114063-01""","""chr2""",212288966,"""C""","""T""","""ERBB4""",true,1229,"""2780G>A""","""R927Q""",0.0033,"""missense""","""NM_005235""","""-""",false,"""1-snv""","""N""","""CTAATAAATCAGGGATTTCTTGCGTTGGAA…",144,38,14,0,90,133,71,242,133,0.027047,0.034211,0.042105,0.0,0.368421,0.066392,1.7303e-8,3.0432e-8,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1151545-01""","""chr22""",29090061,"""G""","""A""","""CHEK2""",true,3097,"""1420C>T""","""R474C""",0.01,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GGCTTCTTCTGTCGTAAAACATGCCTTTGG…",144,328,54,0,141,142,71,160,173,0.008469,0.009146,0.015244,0.567073,0.164634,0.041883,0.039788,0.021899,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1219316-01""","""chr16""",396589,"""C""","""T""","""AXIN1""",false,503,"""437G>A""","""R146Q""",0.0179,"""missense""","""NM_003502""","""-""",false,"""1-snv""","""N""","""TGTTATCAAGAATGTACTTTTGGTAGATGG…",49,22,0,0,41,42,15,184,150,0.005566,0.004545,0.009091,0.0,0.0,1.0459e-7,0.005471,0.005471,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1245811-01""","""chr8""",117878929,"""G""","""C""","""RAD21""",true,1039,"""40C>G""","""L14V""",0.0058,"""missense""","""NM_006265""","""-""",false,"""1-snv""","""N""","""CGCTAGCCAAATTTTGGCCACAGGCCCTCT…",144,47,6,0,95,138,70,172,138,0.007979,0.014894,0.010638,0.0,0.12766,0.069387,1.0670e-9,0.070989,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2115040-01""","""chr16""",3778242,"""C""","""T""","""CREBBP""",true,129,"""6806G>A""","""G2269E""",0.0853,"""missense""","""NM_004380""","""-""",false,"""1-snv""","""N""","""GCCCCATCTGGCCAAGCTGTTCCATCTGAG…",101,19,0,0,90,79,21,90,79,0.007817,0.005263,0.010526,0.0,0.0,1.4551e-12,0.000022,0.000023,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2115040-01""","""chrX""",63411333,"""C""","""T""","""FAM123B""",true,136,"""1834G>A""","""E612K""",0.1103,"""missense""","""NM_152424""","""-""",false,"""1-snv""","""Y""","""AGCTTCCCAAGTGTGGGCCTTCCTGGCATA…",101,22,0,0,17,83,17,17,83,0.020702,0.009091,0.018182,0.0,0.0,2.4810e-33,5.0376e-38,5.0376e-38,true,false,tr

136

In [19]:
summarize_res(xml_df)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",0.763924,165,21599
"""filter_2_hairpin_structure""",0.0,0,21599
"""filter_3_microhomology_induced…",0.648178,140,21599
"""filter_4_highly_homologous_reg…",0.208343,45,21599
"""filter_5_soft_clipped_reads""",0.717626,155,21599
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.00926,2,21599
"""filter_8_low_quality""",1.824159,394,21599
"""msec_filter_123""",1.101903,238,21599


In [20]:
summarize_res(xml_arti_filter_1234)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",38.509317,62,161
"""filter_2_hairpin_structure""",0.0,0,161
"""filter_3_microhomology_induced…",47.826087,77,161
"""filter_4_highly_homologous_reg…",27.950311,45,161
"""filter_5_soft_clipped_reads""",9.31677,15,161
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.0,0,161
"""filter_8_low_quality""",3.10559,5,161
"""msec_filter_123""",72.049689,116,161


In [21]:
summarize_res(xml_arti_all_filter)

filter,percentage,n_failed,total_variants
str,f64,i64,i32
"""filter_1_mutation_intra_hairpi…",7.345972,62,844
"""filter_2_hairpin_structure""",0.0,0,844
"""filter_3_microhomology_induced…",9.123223,77,844
"""filter_4_highly_homologous_reg…",5.331754,45,844
"""filter_5_soft_clipped_reads""",14.810427,125,844
…,…,…,…
"""filter_7_mutation_at_homopolym…",0.236967,2,844
"""filter_8_low_quality""",44.78673,378,844
"""msec_filter_123""",13.744076,116,844
